# STN Flow Field — Geometric Warping Preprocessor

Standalone notebook. Trains a Spatial Transformer Network that learns a dense flow field
to warp ground-level images closer to the satellite domain, reducing the domain gap
before VQGAN encoding.

Pipeline: `Ground → STN (frozen after training) → Warped Ground → VQGAN encoder → Transformer`

In [ ]:
!pip install lpips pytorch-msssim -q

# Configuration

In [ ]:
DATA_ROOT  = "/kaggle/input/datasets/carotenutoalessandro/cvusa-groundandpolar-subset-35191"
SAVE_DIR   = "/kaggle/working"

BATCH_SIZE    = 16
NUM_EPOCHS    = 30
LEARNING_RATE = 1e-4

LPIPS_WEIGHT = 1.0
SSIM_WEIGHT  = 0.5
L1_WEIGHT    = 0.1

print("Config ready")

# Imports

In [ ]:
import os
import csv
import random
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torchvision.transforms.functional as TF
from PIL import Image

import lpips
from pytorch_msssim import ssim

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# Dataset

In [ ]:
class CVUSADataset(Dataset):
    def __init__(self, csv_path, data_root, size=256, is_train=True):
        self.data_root = data_root
        self.is_train  = is_train
        self.file_pairs = []

        print(f"Loading dataset from: {csv_path}")
        with open(csv_path, 'r', newline='', encoding='utf-8') as f:
            reader = csv.reader(f)
            next(reader)  # skip header
            for row in reader:
                if len(row) < 2:
                    continue
                polar_path  = os.path.join(data_root, row[0].replace('\\', os.sep).strip())
                ground_path = os.path.join(data_root, row[1].replace('\\', os.sep).strip())
                if os.path.exists(polar_path) and os.path.exists(ground_path):
                    self.file_pairs.append((polar_path, ground_path))

        print(f"Found {len(self.file_pairs)} valid pairs.")

        self.transform = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
        ])

    def __len__(self):
        return len(self.file_pairs)

    def __getitem__(self, idx):
        polar_path, ground_path = self.file_pairs[idx]
        polar_img  = Image.open(polar_path).convert('RGB')
        ground_img = Image.open(ground_path).convert('RGB')

        if self.is_train:
            if random.random() > 0.5:
                polar_img  = TF.hflip(polar_img)
                ground_img = TF.hflip(ground_img)
            if random.random() > 0.2:
                ground_img = transforms.ColorJitter(0.1, 0.1, 0.1)(ground_img)

        return {
            "satellite": self.transform(polar_img),
            "ground":    self.transform(ground_img)
        }


train_dataset = CVUSADataset(os.path.join(DATA_ROOT, "train.csv"), DATA_ROOT, is_train=True)
val_dataset   = CVUSADataset(os.path.join(DATA_ROOT, "val.csv"),   DATA_ROOT, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# Baseline: LPIPS Ground vs Satellite

Quantifies the raw domain gap — upper bound on how much the STN can help.

In [ ]:
lpips_fn = lpips.LPIPS(net='vgg').to(DEVICE)
lpips_fn.eval()

baseline_scores = []
baseline_ssim_scores = []

with torch.no_grad():
    for batch in val_loader:
        ground    = batch['ground'].to(DEVICE)
        satellite = batch['satellite'].to(DEVICE)

        scores = lpips_fn(ground, satellite).flatten()
        baseline_scores.extend(scores.cpu().tolist())

        ground_01    = (ground + 1) / 2
        satellite_01 = (satellite + 1) / 2
        ssim_val = ssim(ground_01, satellite_01, data_range=1.0, size_average=False)
        baseline_ssim_scores.extend(ssim_val.cpu().tolist())

baseline_lpips = np.mean(baseline_scores)
baseline_ssim  = np.mean(baseline_ssim_scores)
print(f"Baseline LPIPS (ground vs satellite): {baseline_lpips:.4f} ± {np.std(baseline_scores):.4f}")
print(f"Baseline SSIM  (ground vs satellite): {baseline_ssim:.4f} ± {np.std(baseline_ssim_scores):.4f}")

# STN Model

MobileNetV2 (frozen) encoder → upsampling decoder → dense flow field (256×256×2) → grid_sample

In [ ]:
class FlowFieldSTN(nn.Module):
    def __init__(self):
        super().__init__()

        # Encoder: MobileNetV2 pretrained, frozen — output 8×8×1280 for 256×256 input
        mobilenet = models.mobilenet_v2(pretrained=True)
        self.encoder = mobilenet.features
        for p in self.encoder.parameters():
            p.requires_grad = False

        # Decoder: 8×8 → 256×256, output flow field
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(1280, 512, 4, 2, 1), nn.ReLU(),  # 16×16
            nn.ConvTranspose2d(512,  256, 4, 2, 1), nn.ReLU(),  # 32×32
            nn.ConvTranspose2d(256,  128, 4, 2, 1), nn.ReLU(),  # 64×64
            nn.ConvTranspose2d(128,   64, 4, 2, 1), nn.ReLU(),  # 128×128
            nn.ConvTranspose2d( 64,   32, 4, 2, 1), nn.ReLU(),  # 256×256
            nn.Conv2d(32, 2, 3, 1, 1),                          # 256×256×2
            nn.Tanh()
        )

        # Init last conv near zero → identity transform at start
        for m in self.decoder.modules():
            if isinstance(m, nn.Conv2d) and m.out_channels == 2:
                nn.init.zeros_(m.weight)
                nn.init.zeros_(m.bias)

        # Identity sampling grid
        grid_y, grid_x = torch.meshgrid(
            torch.linspace(-1, 1, 256),
            torch.linspace(-1, 1, 256),
            indexing='ij'
        )
        identity_grid = torch.stack([grid_x, grid_y], dim=-1).unsqueeze(0)  # 1×256×256×2
        self.register_buffer('identity_grid', identity_grid)

    def forward(self, ground):
        features     = self.encoder(ground)                           # B×1280×8×8
        displacement = self.decoder(features)                         # B×2×256×256
        displacement = displacement.permute(0, 2, 3, 1) * 0.5        # B×256×256×2, max ±0.5

        sampling_grid = (self.identity_grid + displacement).clamp(-1, 1)
        warped = F.grid_sample(ground, sampling_grid,
                               mode='bilinear', padding_mode='border', align_corners=True)
        return warped


model = FlowFieldSTN().to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")

# Loss

`L = 1.0 × LPIPS + 0.5 × (1 − SSIM) + 0.1 × L1`

In [ ]:
class WarpingLoss(nn.Module):
    def __init__(self, lpips_weight=1.0, ssim_weight=0.5, l1_weight=0.1):
        super().__init__()
        self.lpips_fn     = lpips.LPIPS(net='vgg')
        self.lpips_weight = lpips_weight
        self.ssim_weight  = ssim_weight
        self.l1_weight    = l1_weight

    def forward(self, warped, target):
        l_lpips = self.lpips_fn(warped, target).mean()

        warped_01 = (warped + 1) / 2
        target_01 = (target + 1) / 2
        l_ssim = 1 - ssim(warped_01, target_01, data_range=1.0, size_average=True)

        l_l1 = F.l1_loss(warped, target)

        return self.lpips_weight * l_lpips + self.ssim_weight * l_ssim + self.l1_weight * l_l1


criterion = WarpingLoss(LPIPS_WEIGHT, SSIM_WEIGHT, L1_WEIGHT).to(DEVICE)

# Training

In [ ]:
optimizer = optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

best_val_lpips  = float('inf')
best_model_path = None
os.makedirs(SAVE_DIR, exist_ok=True)

LOG_EVERY = 100  # print every N batches

history = {
    'train_loss':    [],
    'val_loss':      [],
    'val_lpips':     [],
    'val_ssim_loss': [],
    'val_l1':        [],
    'lr':            [],
}

for epoch in range(1, NUM_EPOCHS + 1):

    # --- Train ---
    model.train()
    train_loss = 0.0
    for i, batch in enumerate(train_loader):
        ground    = batch['ground'].to(DEVICE)
        satellite = batch['satellite'].to(DEVICE)

        optimizer.zero_grad()
        warped = model(ground)
        loss   = criterion(warped, satellite)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

        if (i + 1) % LOG_EVERY == 0:
            print(f"  Epoch {epoch} | Batch {i+1}/{len(train_loader)} | Loss: {loss.item():.4f}")

    train_loss /= len(train_loader)

    # --- Eval ---
    print(f"  Epoch {epoch} — evaluating on val set...")
    model.eval()
    val_lpips_scores = []
    val_ssim_scores  = []
    val_l1_scores    = []
    with torch.no_grad():
        for batch in val_loader:
            ground    = batch['ground'].to(DEVICE)
            satellite = batch['satellite'].to(DEVICE)
            warped    = model(ground)

            lpips_scores = criterion.lpips_fn(warped, satellite).flatten()
            val_lpips_scores.extend(lpips_scores.cpu().tolist())

            warped_01    = (warped + 1) / 2
            satellite_01 = (satellite + 1) / 2
            ssim_scores  = ssim(warped_01, satellite_01, data_range=1.0, size_average=False)
            val_ssim_scores.extend(ssim_scores.cpu().tolist())

            val_l1_scores.append(F.l1_loss(warped, satellite).item())

    val_lpips     = np.mean(val_lpips_scores)
    val_ssim_mean = np.mean(val_ssim_scores)
    val_ssim_loss = 1 - val_ssim_mean
    val_l1        = np.mean(val_l1_scores)
    val_loss      = LPIPS_WEIGHT * val_lpips + SSIM_WEIGHT * val_ssim_loss + L1_WEIGHT * val_l1
    current_lr    = optimizer.param_groups[0]['lr']

    scheduler.step(val_lpips)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_lpips'].append(val_lpips)
    history['val_ssim_loss'].append(val_ssim_loss)
    history['val_l1'].append(val_l1)
    history['lr'].append(current_lr)

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} — Train: {train_loss:.4f} | Val: {val_loss:.4f} "
          f"(LPIPS {val_lpips:.4f}  SSIM {val_ssim_loss:.4f}  L1 {val_l1:.4f}) | LR: {current_lr:.2e}")

    if val_lpips < best_val_lpips:
        best_val_lpips = val_lpips
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        best_model_path = os.path.join(SAVE_DIR, f"stn_best_epoch{epoch}_lpips{val_lpips:.4f}_{timestamp}.pth")
        torch.save(model.state_dict(), best_model_path)
        print(f"  ✅ Best saved: {best_model_path}")

print(f"\nTraining complete.")
print(f"Best Val LPIPS: {best_val_lpips:.4f} (baseline was {baseline_lpips:.4f})")
print(f"Best model: {best_model_path}")

# Results — Baseline vs STN

In [ ]:
# Load best model and compute final metrics on full val set
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.eval()

final_lpips_scores = []
final_ssim_scores  = []

with torch.no_grad():
    for batch in val_loader:
        ground    = batch['ground'].to(DEVICE)
        satellite = batch['satellite'].to(DEVICE)
        warped    = model(ground)

        scores = lpips_fn(warped, satellite).flatten()
        final_lpips_scores.extend(scores.cpu().tolist())

        warped_01    = (warped + 1) / 2
        satellite_01 = (satellite + 1) / 2
        ssim_scores  = ssim(warped_01, satellite_01, data_range=1.0, size_average=False)
        final_ssim_scores.extend(ssim_scores.cpu().tolist())

final_lpips = np.mean(final_lpips_scores)
final_ssim  = np.mean(final_ssim_scores)

print("=" * 50)
print(f"{'Metric':<10} {'Baseline':>12} {'STN':>12} {'Delta':>12}")
print("-" * 50)
print(f"{'LPIPS':<10} {baseline_lpips:>12.4f} {final_lpips:>12.4f} {final_lpips - baseline_lpips:>+12.4f}")
print(f"{'SSIM':<10} {baseline_ssim:>12.4f} {final_ssim:>12.4f} {final_ssim - baseline_ssim:>+12.4f}")
print("=" * 50)

# Training Curves

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

epochs = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('STN Training Curves', fontsize=14, fontweight='bold')

# --- Top left: Train loss ---
ax = axes[0, 0]
ax.plot(epochs, history['train_loss'], color='steelblue', linewidth=2, label='Train Loss')
ax.set_title('Train Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Top right: Val loss (total + components) ---
ax = axes[0, 1]
ax.plot(epochs, history['val_loss'],      color='black',      linewidth=2.5, label='Val Loss (total)')
ax.plot(epochs, history['val_lpips'],     color='tomato',     linewidth=1.5, linestyle='--', label=f'LPIPS ×{LPIPS_WEIGHT}')
ax.plot(epochs, history['val_ssim_loss'], color='mediumseagreen', linewidth=1.5, linestyle='--', label=f'1−SSIM ×{SSIM_WEIGHT}')
ax.plot(epochs, history['val_l1'],        color='mediumpurple',   linewidth=1.5, linestyle='--', label=f'L1 ×{L1_WEIGHT}')
ax.set_title('Val Loss — Total + Components')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Bottom left: Learning rate ---
ax = axes[1, 0]
ax.plot(epochs, history['lr'], color='darkorange', linewidth=2)
ax.set_title('Learning Rate')
ax.set_xlabel('Epoch')
ax.set_ylabel('LR')
ax.set_yscale('log')
ax.yaxis.set_major_formatter(ticker.ScalarFormatter())
ax.grid(True, alpha=0.3)

# --- Bottom right: Val LPIPS vs baseline ---
ax = axes[1, 1]
ax.plot(epochs, history['val_lpips'], color='tomato', linewidth=2, label='Val LPIPS (STN)')
ax.axhline(y=baseline_lpips, color='gray', linewidth=1.5, linestyle='--', label=f'Baseline LPIPS ({baseline_lpips:.4f})')
ax.fill_between(epochs, history['val_lpips'], baseline_lpips,
                where=[v < baseline_lpips for v in history['val_lpips']],
                alpha=0.15, color='green', label='Gap closed')
ax.set_title('Val LPIPS vs Baseline Domain Gap')
ax.set_xlabel('Epoch')
ax.set_ylabel('LPIPS')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'stn_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()